# Budget Auction — Experiment 0: Single Buyer, Independent Types

**Theorem 4.2 test bed.** One buyer, one good, $T=5$ stages, types drawn i.i.d. from $\text{Uniform}(0,1)$.
This is the exact canonical setting of Deng et al. (2021) main body where GBAM sufficiency is theoretically guaranteed.

**Expected result:** Variant A (GBAM inputs) should perform on par with B and C.
Any gap remaining is a pure computational artifact, establishing a baseline for how much of the Exp 1/2 gap is optimization noise.

Set `RUN_ENV = 'colab'` or `'local'`.

In [ ]:
RUN_ENV = 'colab'   # 'colab' | 'local'
LOCAL_PROJECT_DIR = None
# LOCAL_PROJECT_DIR = r'E:\Research\code\Dynamic Mechanism Design\Budget Auction'

In [ ]:
import os, sys, subprocess

if RUN_ENV == 'colab':
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'uninstall', '-y',
         'jax', 'jaxlib', 'jax-cuda12-plugin', 'jax-cuda12-pjrt'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '-qU',
         'jax[cuda12]', 'dm-haiku', 'optax', 'flax'])
    print('Install done. IMPORTANT: Runtime \u2192 Restart session, then re-run from this cell.')
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        os.chdir('/content/drive/MyDrive/My Research/Dynamic Mechanism Design/Budget Auction')
        print(f'Working directory: {os.getcwd()}')
    except ImportError:
        print('(Drive mount skipped \u2014 restart first.)')
elif RUN_ENV == 'local':
    if LOCAL_PROJECT_DIR:
        os.chdir(LOCAL_PROJECT_DIR)
    print(f'Working directory: {os.getcwd()}')

In [ ]:
import jax
import jax.numpy as jnp
import haiku as hk
import optax
import matplotlib.pyplot as plt
import pickle, json, os

from budget_auction.core import (
    BudgetSimulator, BoundedContPolicy,
    build_bounded_policy_net, plot_loss_dict,
)
from budget_auction.mechanisms.budget_mechanism import BudgetMechanism
from budget_auction.mechanisms.input_adapters import VARIANT_NETWORK
from budget_auction.envs.single_buyer_indep_env import create_single_buyer_indep_auction
from budget_auction.train.run_acceptance import nested_GD_budget
from budget_auction.train.evaluate import eval_budget_mechanism_multikey, print_eval_report

print('JAX devices:', jax.devices())

## Configuration

In [ ]:
VARIANT = 'A'   # 'A' | 'B' | 'C' | 'A_wide' | 'A_cheat'
SEED    = 42

cfg = dict(
    n_goods=1, horizon=5, discount=0.9,
    initial_budget=3.0,
    profit_goal=1.0,         # starting point; Stage 2 search will refine
    num_outer_epochs=5000,
    num_inner_steps=300,
    lr_theta=1e-4, lr_w=3e-4,
    num_samples_train=128,
    alpha_weight=5.0, beta_weight=0.0,   # IR hard by construction
    print_iter=100,
    policy_hidden=128, residual_init_std=1.0,
)
print('Config:', cfg)

## Build Environment

In [ ]:
auction = create_single_buyer_indep_auction(
    n_goods=cfg['n_goods'], horizon=cfg['horizon'],
    discount=cfg['discount'], initial_budget=cfg['initial_budget'],
    types_low=0.0, types_high=1.0, seed=SEED,
)
print(f'Auction: {auction.num_buyers}B x {auction.num_goods}G x T={auction.num_horizons}')
print(f'Initial budgets: {auction.initial_budgets}')

s0 = auction.get_initial_state(jax.random.PRNGKey(0))
print(f'alpha={s0.alpha}  beta={s0.beta}  types={s0.current_type}')

## Quick Single-Variant Training

In [ ]:
_mnet = hk.without_apply_rng(hk.transform(VARIANT_NETWORK[VARIANT]))
_pnet = build_bounded_policy_net(
    d_priv=auction.d_priv, d_pub=auction.d_pub, G=auction.num_goods,
    hidden=cfg['policy_hidden'], residual_init_std=cfg['residual_init_std'],
)
logs = nested_GD_budget(
    auction=auction, mechanism_network=_mnet, policy_network=_pnet,
    profit_goal=cfg['profit_goal'], num_outer_epochs=cfg['num_outer_epochs'],
    lr_theta=cfg['lr_theta'], num_inner_steps=cfg['num_inner_steps'],
    lr_w=cfg['lr_w'], num_samples_train=cfg['num_samples_train'],
    alpha_weight=cfg['alpha_weight'], beta_weight=cfg['beta_weight'],
    print_iter=cfg['print_iter'], seed=SEED,
)
mechanism_params = logs.pop('mechanism_params')
policy_params    = logs.pop('policy_params')
print('Training complete.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(logs['reg_plus']); axes[0].set_title(f'Regret+ ({VARIANT})')
axes[1].plot(logs['gap_plus']); axes[1].set_title('Profit Gap+')
axes[2].plot(logs['ir']);        axes[2].set_title('IR Loss')
for _ax in axes: _ax.set_xlabel('Epoch')
plt.suptitle(f'Exp 0 (single buyer) \u2014 Variant {VARIANT}', y=1.02)
plt.tight_layout(); plt.show()

## Hyperparameter Search — 3-Stage Pipeline

Same staged protocol as Exp 1/2. For single-buyer, revenue scale is roughly half of Exp 1 (one buyer vs two).

| Stage | Free parameters | Trials |
|-------|----------------|--------|
| **1** | `lr_theta`, `lr_w` | 12 |
| **2** | `profit_goal`, `alpha_weight` | 12 |
| **3** | `num_inner_steps` | 4 |

In [ ]:
import itertools, random, time

HP_TIME_BUDGET_HOURS = 8
HP_S1_EPOCHS = 300
HP_S2_EPOCHS = 500
HP_S3_EPOCHS = 500
HP_SAMPLES   = 64
HP_EVAL_KEYS = 16
HP_EVAL_SAMPS = 128
HP_IC_THRESH = 0.05

hp_save_dir = 'outputs/single_buyer/hparam_search'
os.makedirs(hp_save_dir, exist_ok=True)

def _hp_score(r):
    ic_pen = 10.0 * max(0.0, r.get('regret_mean', 0.0) - HP_IC_THRESH)
    return r['revenue_mean'] - ic_pen

def _run_trial(trial_cfg, trial_seed, n_epochs, n_samples):
    _mnet = hk.without_apply_rng(hk.transform(VARIANT_NETWORK[VARIANT]))
    _pnet = build_bounded_policy_net(
        d_priv=auction.d_priv, d_pub=auction.d_pub,
        G=auction.num_goods, hidden=cfg['policy_hidden'],
        residual_init_std=cfg['residual_init_std'],
    )
    _tl = nested_GD_budget(
        auction=auction, mechanism_network=_mnet, policy_network=_pnet,
        profit_goal=trial_cfg['profit_goal'],
        num_outer_epochs=n_epochs,
        lr_theta=trial_cfg['lr_theta'],
        num_inner_steps=trial_cfg['num_inner_steps'],
        lr_w=trial_cfg['lr_w'],
        num_samples_train=n_samples,
        alpha_weight=trial_cfg['alpha_weight'],
        beta_weight=0.0,
        print_iter=10**7, seed=trial_seed,
    )
    _mp = _tl.pop('mechanism_params'); _pp = _tl.pop('policy_params')
    _ek = jax.random.split(jax.random.PRNGKey(trial_seed + 99999), HP_EVAL_KEYS)
    _ev = eval_budget_mechanism_multikey(
        auction=auction, mechanism_network=_mnet, mechanism_params=_mp,
        profit_goal=trial_cfg['profit_goal'],
        num_samples=HP_EVAL_SAMPS, keys=_ek,
        policy_network=_pnet, policy_params=_pp,
    )
    return {
        'config':       trial_cfg,
        'revenue_mean': float(_ev['profit_stats']['mean']),
        'regret_mean':  float(_ev.get('regret_stats', {}).get('mean', float('nan'))),
        'seed': trial_seed,
    }

def _run_stage(stage_name, combos, fixed_cfg, n_epochs, n_samples, results_path, start, budget_s):
    _results = json.load(open(results_path)) if os.path.exists(results_path) else []
    _done = {str(r['config']) for r in _results}
    for _i, _tcfg in enumerate(combos):
        _full = {**fixed_cfg, **_tcfg}
        if str(_full) in _done: continue
        if time.time() - start >= budget_s:
            print(f'  [{stage_name}] time budget reached'); break
        print(f'  [{stage_name}] {_i+1}/{len(combos)} | {_tcfg}')
        try:
            _rec = _run_trial(_full, SEED + _i, n_epochs, n_samples)
            _rec['score'] = _hp_score(_rec)
            _results.append(_rec); _done.add(str(_full))
            json.dump(_results, open(results_path, 'w'), indent=2)
            print(f'    => rev={_rec["revenue_mean"]:.4f}  score={_rec["score"]:.4f}')
        except Exception as _e:
            print(f'    [FAILED] {_e}')
    return sorted(_results, key=lambda r: r['score'], reverse=True)

_start = time.time(); _budget_s = HP_TIME_BUDGET_HOURS * 3600
print(f'Search started. Budget: {HP_TIME_BUDGET_HOURS}h')

In [ ]:
# Stage 1: Learning rates
_s1_search = {'lr_theta': [3e-5, 1e-4, 3e-4, 1e-3], 'lr_w': [1e-4, 3e-4, 1e-3]}
_s1_fixed  = {'alpha_weight': 5.0, 'profit_goal': 1.0, 'num_inner_steps': 150}
_s1_combos = [dict(zip(_s1_search, c)) for c in itertools.product(*_s1_search.values())]
random.seed(SEED); random.shuffle(_s1_combos)

_s1_results = _run_stage('S1-LR', _s1_combos, _s1_fixed, HP_S1_EPOCHS, HP_SAMPLES,
                          os.path.join(hp_save_dir, 's1_lr.json'), _start, _budget_s)
lr_theta_star = _s1_results[0]['config']['lr_theta']
lr_w_star     = _s1_results[0]['config']['lr_w']
print(f'Stage 1 best: lr_theta={lr_theta_star:.1e}  lr_w={lr_w_star:.1e}')

In [ ]:
# Stage 2: Profit goal x alpha_weight
_s2_search = {'profit_goal': [0.7, 0.9, 1.1, 1.3], 'alpha_weight': [1.0, 5.0, 10.0]}
_s2_fixed  = {'lr_theta': lr_theta_star, 'lr_w': lr_w_star, 'num_inner_steps': 200}
_s2_combos = [dict(zip(_s2_search, c)) for c in itertools.product(*_s2_search.values())]

_s2_results = _run_stage('S2-Weights', _s2_combos, _s2_fixed, HP_S2_EPOCHS, HP_SAMPLES,
                          os.path.join(hp_save_dir, 's2_weights.json'), _start, _budget_s)
profit_goal_star  = _s2_results[0]['config']['profit_goal']
alpha_weight_star = _s2_results[0]['config']['alpha_weight']
print(f'Stage 2 best: profit_goal={profit_goal_star}  alpha_weight={alpha_weight_star}')

In [ ]:
# Stage 3: Inner steps
_s3_search = {'num_inner_steps': [100, 200, 300, 500]}
_s3_fixed  = {'lr_theta': lr_theta_star, 'lr_w': lr_w_star,
               'profit_goal': profit_goal_star, 'alpha_weight': alpha_weight_star}
_s3_combos = [{'num_inner_steps': n} for n in _s3_search['num_inner_steps']]

_s3_results = _run_stage('S3-Inner', _s3_combos, _s3_fixed, HP_S3_EPOCHS, HP_SAMPLES,
                          os.path.join(hp_save_dir, 's3_inner.json'), _start, _budget_s)
inner_steps_star = _s3_results[0]['config']['num_inner_steps']
print(f'Stage 3 best: num_inner_steps={inner_steps_star}')

best_cfg_hp = {
    'lr_theta': lr_theta_star, 'lr_w': lr_w_star,
    'profit_goal': profit_goal_star, 'alpha_weight': alpha_weight_star,
    'num_inner_steps': inner_steps_star,
}
print('\nOptimal config:', best_cfg_hp)

## Final Training with Optimal Hyperparameters

In [ ]:
final_mnet = hk.without_apply_rng(hk.transform(VARIANT_NETWORK[VARIANT]))
final_pnet = build_bounded_policy_net(
    d_priv=auction.d_priv, d_pub=auction.d_pub,
    G=auction.num_goods, hidden=cfg['policy_hidden'],
    residual_init_std=cfg['residual_init_std'],
)
final_logs = nested_GD_budget(
    auction=auction, mechanism_network=final_mnet, policy_network=final_pnet,
    profit_goal=best_cfg_hp['profit_goal'],
    num_outer_epochs=cfg['num_outer_epochs'],
    lr_theta=best_cfg_hp['lr_theta'],
    num_inner_steps=best_cfg_hp['num_inner_steps'],
    lr_w=best_cfg_hp['lr_w'],
    num_samples_train=cfg['num_samples_train'],
    alpha_weight=best_cfg_hp['alpha_weight'],
    beta_weight=0.0,
    print_iter=cfg['print_iter'], seed=SEED,
)
final_mechanism_params = final_logs.pop('mechanism_params')
final_policy_params    = final_logs.pop('policy_params')
print('Final training complete.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(final_logs['reg_plus']); axes[0].set_title('Regret+')
axes[1].plot(final_logs['gap_plus']); axes[1].set_title('Profit Gap+')
axes[2].plot(final_logs['ir']);        axes[2].set_title('IR Loss')
for _ax in axes: _ax.set_xlabel('Epoch')
plt.suptitle(f'Exp 0 \u2014 Variant {VARIANT} (optimal hparams)', y=1.02)
plt.tight_layout(); plt.show()

final_eval_keys = jax.random.split(jax.random.PRNGKey(88888), 32)
final_results = eval_budget_mechanism_multikey(
    auction=auction, mechanism_network=final_mnet,
    mechanism_params=final_mechanism_params,
    profit_goal=best_cfg_hp['profit_goal'],
    num_samples=256, keys=final_eval_keys,
    policy_network=final_pnet, policy_params=final_policy_params,
)
print_eval_report(final_results, prefix=f'Exp 0 (single buyer) Variant {VARIANT}: ')

## Cross-Variant Comparison — GBAM Sufficiency Test

Train all five variants under identical optimal hyperparameters.

**Key comparisons:**
- **A vs C**: If A \u2248 C, GBAM sufficiency holds computationally. If A < C, there is a computational gap even in the theorem's scope.
- **A_wide vs A**: Capacity effect (should be near zero if A is not capacity-limited).
- **A_cheat vs C**: Upper bound on the information value of knowing true types directly.

**Interpretation**: Any A/C gap here serves as a calibration for interpreting the larger gaps in Exp 1 and Exp 2.

In [ ]:
variant_results = {
    VARIANT: {'logs': final_logs, 'eval': final_results,
              'mech_params': final_mechanism_params,
              'pol_params':  final_policy_params}
}

_all_variants = ['A', 'B', 'C', 'A_wide', 'A_cheat']
_variant_colors = {
    'A': 'tab:blue', 'B': 'tab:orange', 'C': 'tab:green',
    'A_wide': 'tab:purple', 'A_cheat': 'tab:red',
}

for _v in _all_variants:
    if _v == VARIANT:
        continue
    print(f'\n--- Variant {_v} ---')
    if _v == 'A_cheat':
        print('  NOTE: A_cheat receives true v_t \u2014 not IC-implementable. Revenue = upper bound only.')
    _vnet  = hk.without_apply_rng(hk.transform(VARIANT_NETWORK[_v]))
    _vpnet = build_bounded_policy_net(
        d_priv=auction.d_priv, d_pub=auction.d_pub,
        G=auction.num_goods, hidden=cfg['policy_hidden'],
        residual_init_std=cfg['residual_init_std'],
    )
    _vl = nested_GD_budget(
        auction=auction, mechanism_network=_vnet, policy_network=_vpnet,
        profit_goal=best_cfg_hp['profit_goal'],
        num_outer_epochs=cfg['num_outer_epochs'],
        lr_theta=best_cfg_hp['lr_theta'],
        num_inner_steps=best_cfg_hp['num_inner_steps'],
        lr_w=best_cfg_hp['lr_w'],
        num_samples_train=cfg['num_samples_train'],
        alpha_weight=best_cfg_hp['alpha_weight'],
        beta_weight=0.0,
        print_iter=cfg['print_iter'], seed=SEED,
    )
    _vmp = _vl.pop('mechanism_params'); _vpp = _vl.pop('policy_params')
    _vek = jax.random.split(jax.random.PRNGKey(88888), 32)
    _vev = eval_budget_mechanism_multikey(
        auction=auction, mechanism_network=_vnet, mechanism_params=_vmp,
        profit_goal=best_cfg_hp['profit_goal'],
        num_samples=256, keys=_vek,
        policy_network=_vpnet, policy_params=_vpp,
    )
    print_eval_report(_vev, prefix=f'Variant {_v}: ')
    variant_results[_v] = {'logs': _vl, 'eval': _vev, 'mech_params': _vmp, 'pol_params': _vpp}

In [ ]:
# Summary table
print('\n' + '='*88)
print(f'  Exp 0 (Single Buyer) \u2014 All Variants  (profit_goal={profit_goal_star})')
print('='*88)
print(f'  {"Var":>8} | {"Revenue":>9} | {"Gap+":>7} | {"IC Reg":>8} | {"Surplus":>8} | Note')
print('  ' + '-'*76)
for _v in _all_variants:
    if _v not in variant_results: continue
    _ev = variant_results[_v]['eval']
    _rg = _ev.get('regret_stats', {}).get('mean', float('nan'))
    _note = '\u2190 upper bound, IC n/a' if _v == 'A_cheat' else \
            '\u2190 capacity control'    if _v == 'A_wide'  else ''
    print(f'  {_v:>8} | {_ev["profit_stats"]["mean"]:>9.4f} | '
          f'{_ev["gap_plus_stats"]["mean"]:>7.4f} | '
          f'{_rg:>8.4f} | '
          f'{_ev["ir_loss_stats"]["mean"]:>8.4f} | {_note}')

print('\nKey: A \u2248 C \u21d2 GBAM computationally sufficient;  A < C \u21d2 optimization gap in theorem scope')

# Learning-curve comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for _v in _all_variants:
    if _v not in variant_results: continue
    _l   = variant_results[_v]['logs']
    _col = _variant_colors[_v]
    _ls  = '--' if _v in ('A_cheat', 'A_wide') else '-'
    axes[0].plot(_l['reg_plus'], color=_col, linestyle=_ls, label=_v, alpha=0.85)
    axes[1].plot(_l['gap_plus'], color=_col, linestyle=_ls, label=_v, alpha=0.85)
    axes[2].plot(_l['ir'],       color=_col, linestyle=_ls, label=_v, alpha=0.85)
for _ax, _t in zip(axes, ['Regret+', 'Profit Gap+', 'IR Loss']):
    _ax.set_title(_t); _ax.set_xlabel('Epoch'); _ax.legend(fontsize=8)
plt.suptitle('Exp 0 (Single Buyer) \u2014 All Variants (dashed = diagnostic)', y=1.02)
plt.tight_layout(); plt.show()

## Save Outputs

In [ ]:
save_dir = f'outputs/single_buyer/seed{SEED}/optimal'
os.makedirs(save_dir, exist_ok=True)

variant_summary = {}
for _v, _vr in variant_results.items():
    with open(f'{save_dir}/mech_params_{_v}.pkl', 'wb') as _f:
        pickle.dump(_vr['mech_params'], _f)
    with open(f'{save_dir}/pol_params_{_v}.pkl', 'wb') as _f:
        pickle.dump(_vr['pol_params'], _f)
    with open(f'{save_dir}/logs_{_v}.pkl', 'wb') as _f:
        pickle.dump(_vr['logs'], _f)
    _ev = _vr['eval']
    variant_summary[_v] = {
        'revenue':     _ev['profit_stats'],
        'gap_plus':    _ev['gap_plus_stats'],
        'regret':      _ev.get('regret_stats', {}),
        'optimal_cfg': best_cfg_hp,
    }

with open(f'{save_dir}/variant_comparison.json', 'w') as _f:
    json.dump(variant_summary, _f, indent=2)
with open(f'{save_dir}/optimal_hparams.json', 'w') as _f:
    json.dump({'optimal_config': best_cfg_hp, 'exp': 'Exp 0', 'variant': VARIANT}, _f, indent=2)

print(f'Saved to: {save_dir}/')
print(f'  mech/pol/logs_{{A,B,C,A_wide,A_cheat}}.pkl, variant_comparison.json, optimal_hparams.json')